# pilot_paper 结果闭合入口

该 Notebook 用于在 Colab 中完成 pilot_paper 结果闭合: 挂载 Google Drive、拉取当前仓库、从 ``SLM_WM_DRIVE_RESULT_ROOT`` 物化前序结果包、重建 attack matrix、external baseline formal import、internal ablation、pilot_paper fixed-FPR 共同协议记录, 并把完整结果包写回 Google Drive。

Notebook 只调度 `scripts/` 中的 repository commands, 不直接手写正式 records、tables、figures 或 reports。


## 运行前准备

1. 确认前序方法主流程和 baseline Notebook 已把结果包写入 ``SLM_WM_DRIVE_RESULT_ROOT`/`。
2. 该入口不需要 GPU, 但需要能够访问 Google Drive。
3. 如果本次需要干净复盘, 应先清理 ``SLM_WM_DRIVE_RESULT_ROOT`/` 后重新运行前序 Notebook, 再运行本入口。
4. 输出完整结果包默认写入 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `complete_result_package``。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
paper_run_name = SLM_WM_PAPER_RUN_NAME.strip() or "pilot_paper"
os.environ['SLM_WM_PAPER_RUN_NAME'] = paper_run_name
prompt_file_by_run = {
    'pilot_paper': 'configs/paper_main_pilot_paper_prompts.txt',
    'full_paper': 'configs/paper_main_full_paper_prompts.txt',
}
drive_result_root = f'/content/drive/MyDrive/SLM/{paper_run_name}_results'
os.environ['SLM_WM_DRIVE_RESULT_ROOT'] = drive_result_root
paper_run_sample_count = os.environ.get('SLM_WM_PAPER_RUN_SAMPLE_COUNT', 'all')
os.environ['SLM_WM_PAPER_RUN_SAMPLE_COUNT'] = paper_run_sample_count
prompt_count_by_run = {'pilot_paper': 600, 'full_paper': 6000}
def resolve_paper_run_count(value):
    normalized = str(value).strip().lower()
    if normalized in {'', 'all', 'none', 'unlimited'}:
        return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper']))
    resolved = int(normalized)
    return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper'])) if resolved <= 0 else resolved
paper_run_expected_sample_count = resolve_paper_run_count(paper_run_sample_count)
paper_run_minimum_clean_negative_count = os.environ.get('SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT'] = paper_run_minimum_clean_negative_count
paper_run_dataset_minimum_count = os.environ.get('SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT'] = paper_run_dataset_minimum_count
target_fpr_by_run = {'pilot_paper': '0.01', 'full_paper': '0.001'}
paper_run_target_fpr = os.environ.get('SLM_WM_PAPER_RUN_TARGET_FPR', target_fpr_by_run.get(paper_run_name, target_fpr_by_run['pilot_paper']))
os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'] = paper_run_target_fpr
paper_run_target_fpr_token = paper_run_target_fpr.replace('.', '_')
os.environ['SLM_WM_PROTOCOL_PROFILE'] = f'{paper_run_name}_fixed_fpr_{paper_run_target_fpr_token}'
os.environ['SLM_WM_PROMPT_SET'] = paper_run_name
os.environ['SLM_WM_PROMPT_FILE'] = prompt_file_by_run.get(paper_run_name, prompt_file_by_run['pilot_paper'])
os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'] = drive_result_root
os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR'] = f'{drive_result_root}/complete_result_package'
os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'] = paper_run_target_fpr
print({
    'protocol_profile': os.environ['SLM_WM_PROTOCOL_PROFILE'],
    'prompt_set': os.environ['SLM_WM_PROMPT_SET'],
    'prompt_file': os.environ['SLM_WM_PROMPT_FILE'],
    'package_search_root': os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'],
    'complete_drive_output_dir': os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR'],
    'target_fpr': os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'],
})


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')
if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)
subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
from pathlib import Path

package_search_root = Path(os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'])
assert package_search_root.exists(), f'未找到当前论文运行层级的 Google Drive 结果目录: {package_search_root}'
zip_count_by_dir = {
    child.name: len(list(child.glob('*.zip')))
    for child in sorted(package_search_root.iterdir())
    if child.is_dir()
}
print({'package_search_root': str(package_search_root), 'zip_count_by_dir': zip_count_by_dir})


In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timezone

from paper_workflow.colab_utils.progress import progress_bar, update_progress

package_search_root = os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT']
complete_drive_output_dir = os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR']
target_fpr = os.environ['SLM_WM_PAPER_RUN_TARGET_FPR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
complete_archive_name = f'pilot_paper_complete_result_package_{utc_suffix}_{short_commit}.zip'

commands = [
    [sys.executable, 'scripts/write_pilot_paper_result_records.py', '--package-search-root', package_search_root, '--materialize-only'],
    [sys.executable, 'scripts/write_attack_matrix_outputs.py'],
    [sys.executable, 'scripts/write_primary_baseline_result_candidates.py', '--target-fpr-override', target_fpr],
    [sys.executable, 'scripts/write_primary_baseline_formal_import_protocol.py'],
    [sys.executable, 'scripts/write_external_baseline_comparison_outputs.py'],
    [sys.executable, 'scripts/write_internal_ablation_outputs.py'],
    [sys.executable, 'scripts/write_pilot_paper_result_records.py', '--require-existing-evidence'],
    [
        sys.executable,
        'scripts/write_pilot_paper_fixed_fpr_common_protocol_outputs.py',
        '--candidate-records-path',
        'outputs/pilot_paper_fixed_fpr_results/pilot_paper_result_records.jsonl',
        '--require-existing-evidence',
    ],
    [
        sys.executable,
        'scripts/write_pilot_paper_complete_result_package.py',
        '--package-search-root',
        package_search_root,
        '--drive-output-dir',
        complete_drive_output_dir,
        '--archive-name',
        complete_archive_name,
        '--skip-package-materialization',
        '--zip-compression',
        'stored',
    ],
]

with progress_bar(len(commands), desc='pilot_paper closure commands', enabled=True) as command_progress:
    for command_index, command in enumerate(commands, start=1):
        command_name = command[1] if len(command) > 1 else command[0]
        print('run_repository_command', ' '.join(command))
        subprocess.run(command, check=True)
        update_progress(
            command_progress,
            profile=f'command={command_name} index={command_index}/{len(commands)}',
        )


In [ ]:
from pathlib import Path

summary_paths = [
    Path('outputs/pilot_paper_fixed_fpr_results/pilot_paper_result_record_summary.json'),
    Path('outputs/pilot_paper_fixed_fpr_common_protocol/pilot_paper_common_protocol_summary.json'),
    Path('outputs/pilot_paper_complete_result_package/pilot_paper_complete_package_summary.json'),
]
for summary_path in summary_paths:
    print(summary_path)
    print(summary_path.read_text(encoding='utf-8')[:4000])

complete_archives = sorted(Path(os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR']).glob('pilot_paper_complete_result_package_*.zip'))
assert complete_archives, '未找到写回 Google Drive 的 当前论文运行层级完整结果包'
print({'latest_complete_archive': str(complete_archives[-1])})
